# Building the fine-tuning dataset

Notebooks 1-4 taught the model *language*. Its corpus was one continuous stream of text —
every message from every chat run together with spaces — so the only thing it could learn is
which token tends to follow which. It has no idea who is speaking, and no idea that a turn
ever *ends*: ask the pre-trained model to generate and it keeps going until `max_new_tokens`
runs out, mid-sentence.

This notebook rebuilds the same chats as discrete, labelled samples:

```
<|startoftext|>Riya<|separator|>hey, are you coming?<|endoftext|>
```

Same words, three pieces of structure they did not have before:

- `<|startoftext|>` — a sample begins here, and nothing before it is context.
- `<|separator|>` — to its left is *who is speaking*, to its right is what they said. At
  generation time you write everything up to this marker and let the model continue, which
  is how you ask one particular person for a reply.
- `<|endoftext|>` — the turn is over. This is the id passed as `eos_token_id`, and it is
  what lets generation stop by itself instead of always running to the token limit.

| Stage | What happens |
| --- | --- |
| **Clean** | Notebook 1's ten rules, applied to the same exports |
| **Group** | Consecutive messages from one sender collapse into a single turn |
| **Wrap** | Each turn becomes one sample, bounded by the markers above |
| **Verify** | Every sample round-trips through the notebook 2 tokenizer |
| **Save** | A JSON list of strings for the fine-tuning run to load |

**None of this is new data.** Every message below was already in `combined_text.txt`, so the
model has seen all of it. What is new is the *shape*: who said it, and where it stopped.

## Cleaning, again

Notebook 1 already solved this, and its output is not reusable here. `combined_text.txt` is
one string of messages joined by spaces — the sender was dropped on the way out, because
pre-training does not need it. Fine-tuning is entirely *about* the sender, so the exports
have to be read a second time.

Notebooks cannot import each other, so the patterns below are notebook 1's, restated.
(Notebook 4 restates `get_vocab_size` for the same reason.) **If you change a rule, change
it in both places** — otherwise the fine-tuning data stops matching the text the model was
pre-trained on.

One thing is deliberately not carried over: the timestamp. Nothing downstream uses it, and
parsing it was the most error-prone part of notebook 1 — `01/03/2026` is 1 March or
3 January depending on the exporting phone, and guessing wrong corrupts dates with no
exception and no `NaT` to notice. Message order here comes from the file, which WhatsApp
already writes chronologically, so this notebook matches a timestamp only to recognise where
a message starts and then throws it away. No pandas, no format inference, no ambiguity.

In [14]:
import json
import re
import sys
from pathlib import Path

# --- Paths ----------------------------------------------------------------
CHAT_DIRECTORY = Path("../Data/private")           # the same exports notebook 1 reads
TOKENIZER_MODEL = Path("../output/tokenizer/my_tokenizer.model")
OUTPUT_PATH = Path("../output/fine_tuning/data/fine_tuning.json")

# --- The sample format ----------------------------------------------------
# The literal strings notebook 2 registered. Their ids are read back off the saved
# tokenizer further down rather than written here, so the two cannot drift apart.
START_OF_TEXT = "<|startoftext|>"
SEPARATOR = "<|separator|>"
END_OF_TEXT = "<|endoftext|>"
MARKERS = (START_OF_TEXT, SEPARATOR, END_OF_TEXT)

BLOCK_SIZE = 256   # notebook 4's context length; longer samples are truncated when training

# --- Cleaning: notebook 1's rules, compiled once --------------------------
# str.translate() applies the whole table in one C-level pass.
UNICODE_FIXES = str.maketrans({
    "\u202f": " ",   # narrow no-break space (iOS puts it before AM/PM)
    "\u00a0": " ",   # non-breaking space
    "\u200e": None,  # left-to-right mark  -> delete
    "\u200f": None,  # right-to-left mark  -> delete
})

# Rules 1-7: one alternation instead of seven checks, so the line is scanned once.
# (In VERBOSE mode whitespace is ignored, so literal spaces are escaped as "\ ".)
DROP_LINE = re.compile(
    r"""
      Messages\ and\ calls\ are\ end-to-end\ encrypted  # 1 encryption notice
    | <Media\ omitted>                                    # 2 attachments
    | [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}       # 3 email address
    | https?://\S+                                        # 4 link
    | You\ deleted\ this\ message                         # 5 deleted message
    | created\ group                                      # 6 group created
    | added\ you                                          # 7 added to group
    """,
    re.VERBOSE,
)

# Rules 9-10: scrub inside the line, then close the gap the scrub left behind.
INLINE_NOISE = re.compile(r"<This message was edited>|@\w+")
EXTRA_SPACE = re.compile(r"[ \t]{2,}")

# Anchored with ^ and matched via .match(), so a continuation line is rejected on
# its very first character. Handles both export flavours:
#   Android:  14/07/2026, 18:30 - Riya: hey
#   iOS:     [15/07/2026, 9:05 AM] ~ Arjun: hey
MESSAGE_HEADER = re.compile(
    r"""^\[?                                  # iOS wraps the timestamp in [ ]
        (\d{1,2}/\d{1,2}/\d{2,4},\s           # date
         \d{1,2}:\d{2}(?::\d{2})?             # time, seconds optional
         (?:\s?[APap][Mm])?)                   # AM/PM, iOS only
        \]?
        \s?[-~]?\s?                            # " - " on Android, " ~ " on iOS
        ([^:]{1,60}):\                           # sender, up to the first colon
        (.*)$                                   # the message itself
    """,
    re.VERBOSE,
)

### One pass, two lists

The reader walks each file once, keeping senders and bodies in parallel lists. Two details
carry over from notebook 1 unchanged:

**Continuation lines.** A line with no timestamp header is not a new message — it is the
rest of the previous one, so it is appended to the last body collected. A continuation that
arrives before any message has nothing to attach to and is dropped.

**Rule 8 runs last.** `null` is junk only when it is the *whole* message; a line like
`is it null or empty?` has to survive, so it cannot be folded into `DROP_LINE` with the
others. That same final pass drops bodies which scrubbing emptied out — a message that was
nothing but an `@mention` leaves an empty string behind, and an empty turn would teach the
model that `<|separator|>` is sometimes followed immediately by `<|endoftext|>`.

In [15]:
def read_chat(file_path) -> list[tuple[str, str]]:
    """Read one WhatsApp export into (sender, message) pairs, in chat order."""
    text = Path(file_path).read_text(encoding="utf-8").translate(UNICODE_FIXES)

    senders: list[str] = []
    bodies: list[str] = []

    for line in text.splitlines():
        if DROP_LINE.search(line):                     # rules 1-7
            continue

        # Scrub inline noise, then close the gaps it left behind (rules 9-10).
        line = EXTRA_SPACE.sub(" ", INLINE_NOISE.sub("", line)).strip()
        if not line:
            continue

        header = MESSAGE_HEADER.match(line)
        if header:
            _, sender, body = header.groups()          # the timestamp is not needed here
            senders.append(sender.strip())
            bodies.append(body.strip())
        elif bodies:
            # No header: this is the next line of the message we are already building.
            bodies[-1] = f"{bodies[-1]}\n{line}"

    # Rule 8, plus anything scrubbing emptied out.
    return [
        (sender, body)
        for sender, body in zip(senders, bodies)
        if body and body.lower() != "null"
    ]

### Checking the rules actually fire

Two of the ten rules never match anything in `../Data/private` — none of these exports
contains a `created group` or `added you` notice — so a normal run tells you nothing about
whether they still work. That is how notebook 1's `null` rule stayed broken for so long: it
compared `line.split(" ")[-1]` against `readlines()` output, which keeps the trailing `\n`,
so it never matched anything at all, and the 18 `null` bodies in these chats went straight
into the training corpus.

The sample below triggers all ten rules at once, plus a multi-line message and an iOS-format
line. Run it whenever you touch a pattern; it costs milliseconds.

In [16]:
import tempfile

SAMPLE = """\
14/07/2026, 18:30 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
14/07/2026, 18:31 - Riya: <Media omitted>
14/07/2026, 18:32 - Riya: mail me at riya@example.com
14/07/2026, 18:33 - Riya: https://www.example.com/page
14/07/2026, 18:34 - Riya: hey, how are you? <This message was edited>
14/07/2026, 18:35 - Riya: You deleted this message
14/07/2026, 18:36 - Riya: null
14/07/2026, 18:37 - Riya created group "study group"
14/07/2026, 18:38 - Riya added you
14/07/2026, 18:39 - Riya: @arjun are you coming?
14/07/2026, 18:40 - Riya: line one
continued on the next line
[15/07/2026, 9:05 AM] ~ Arjun: iOS format works too
"""

EXPECTED = [
    ("Riya", "hey, how are you?"),
    ("Riya", "are you coming?"),
    ("Riya", "line one\ncontinued on the next line"),
    ("Arjun", "iOS format works too"),
]

sample_file = Path(tempfile.mkdtemp()) / "sample.txt"
sample_file.write_text(SAMPLE, encoding="utf-8")

actual = read_chat(sample_file)

assert actual == EXPECTED, "\n".join(
    ["cleaning rules did not behave as documented:"]
    + [f"  expected {row}" for row in EXPECTED]
    + [f"  actual   {row}" for row in actual]
)
print(f"all 10 rules behave as documented ({len(actual)} messages kept out of 13 lines)")

all 10 rules behave as documented (4 messages kept out of 13 lines)


## One turn per sender, not one per message

People send bursts. Three lines in a row from the same person are one thought, not three,
but the export writes each on its own line:

```
User 1: Hey!
User 1: How are you?
User 2: I am fine
User 2: And you?
User 1: Good.
```

Left alone that becomes five samples, each ending in `<|endoftext|>` after a handful of
words — and the model would learn exactly that: turns are tiny, stop early. Merging each run
into one turn gives it the real distribution instead:

```
User 1: Hey!\nHow are you?
User 2: I am fine\nAnd you?
User 1: Good.
```

The newline earns its place: it keeps the original messages visible as separate lines
without pretending they were separate turns.

**Grouping runs per export, never across the corpus.** Two chats sit next to each other in
file order and have nothing to do with one another, so a turn must never span the boundary
between them — which is why the loop further down calls this once per file.

In [4]:
def group_by_sender(messages: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """Merge each run of consecutive messages from one sender into a single turn."""
    # Parts are collected in a list and joined once at the end. Growing the string with
    # += instead would re-copy the whole turn per message: quadratic in run length.
    runs: list[tuple[str, list[str]]] = []

    for sender, message in messages:
        if runs and runs[-1][0] == sender:
            runs[-1][1].append(message)
        else:
            runs.append((sender, [message]))

    return [(sender, "\n".join(parts)) for sender, parts in runs]

### Checking the grouping

The example from the markdown above, asserted rather than described — including the part
that is easy to get wrong: `User 1` speaks again at the end, and that turn must **not** be
merged back into their earlier one. Only *adjacent* runs collapse.

In [5]:
GROUPING_SAMPLE = [
    ("User 1", "Hey!"),
    ("User 1", "How are you?"),
    ("User 2", "I am fine"),
    ("User 2", "And you?"),
    ("User 1", "Good."),
]

GROUPING_EXPECTED = [
    ("User 1", "Hey!\nHow are you?"),
    ("User 2", "I am fine\nAnd you?"),
    ("User 1", "Good."),
]

grouped = group_by_sender(GROUPING_SAMPLE)

assert grouped == GROUPING_EXPECTED, "\n".join(
    ["grouping did not behave as documented:"]
    + [f"  expected {row}" for row in GROUPING_EXPECTED]
    + [f"  actual   {row}" for row in grouped]
)
assert group_by_sender([]) == [], "an empty chat should produce no turns"

print(f"{len(GROUPING_SAMPLE)} messages -> {len(grouped)} turns, "
      f"and the repeated sender was not merged back")

5 messages -> 3 turns, and the repeated sender was not merged back


## Wrapping a turn into a sample

Nothing clever here — the markers are string concatenation:

```
<|startoftext|> + sender + <|separator|> + message + <|endoftext|>
```

What makes it work is that the tokenizer maps each marker to **one id**, so the model sees a
single unambiguous symbol rather than a run of ordinary characters it would have to learn to
recognise as a unit.
`<|padding|>` and `<|unk|>` were registered in notebook 2 as well but do not appear here:
padding is the data loader's job at fine-tuning time, and nothing is ever out-of-vocabulary
because BPE falls back to raw bytes.

In [6]:
def build_sample(sender: str, message: str) -> str:
    """Wrap one turn in the markers the model is being taught to recognise."""
    return f"{START_OF_TEXT}{sender}{SEPARATOR}{message}{END_OF_TEXT}"

### The tokenizer has to agree

The format above is only real if the tokenizer treats the markers as special tokens. If it
does not, `<|separator|>` encodes as ordinary characters, the model never gets a clean
boundary signal, and `eos_token_id` matches nothing — generation runs to the token limit
every time, and the failure looks like a training problem rather than a data problem.

So the ids are read off the saved tokenizer rather than hard-coded, and checked.

In [7]:
try:
    import minbpe  # already installed in this environment
except ModuleNotFoundError:
    here = Path.cwd()
    for repo_root in (here, *here.parents):
        if (repo_root / "minbpe" / "minbpe" / "base.py").exists():
            sys.path.insert(0, str(repo_root / "minbpe"))
            print("using minbpe clone at:", repo_root / "minbpe")
            break
    else:
        raise ModuleNotFoundError(
            "minbpe not found. Install it with "
            "`pip install git+https://github.com/karpathy/minbpe.git`"
        )
else:
    print("using installed minbpe:", Path(minbpe.__file__).parent)

from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(TOKENIZER_MODEL))

missing = [marker for marker in MARKERS if marker not in tokenizer.special_tokens]
assert not missing, f"{missing} are not special tokens - re-run 2_BytePairEncoding.ipynb"

MARKER_IDS = {marker: tokenizer.special_tokens[marker] for marker in MARKERS}
for marker, idx in MARKER_IDS.items():
    print(f"{idx:>6}  {marker}")

using minbpe clone at: /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/minbpe
  1024  <|startoftext|>
  1025  <|separator|>
  1026  <|endoftext|>


### Checking one sample end to end

Four assertions on the smallest possible sample: the markers are single ids, they land in
the right places, the text survives a round-trip, and the two halves can be split apart
again — which is what a fine-tuning run does to turn a sample into a prompt and a target.

In [8]:
probe_sample = build_sample("Riya", "hey, are you coming?")
probe_ids = tokenizer.encode(probe_sample, allowed_special="all")

# 1. the markers are single ids, in the right places
assert probe_ids[0] == MARKER_IDS[START_OF_TEXT], f"first id is {probe_ids[0]}"
assert probe_ids[-1] == MARKER_IDS[END_OF_TEXT], f"last id is {probe_ids[-1]}"
assert probe_ids.count(MARKER_IDS[SEPARATOR]) == 1, "separator is not exactly one id"

# 2. nothing is lost on the way back
assert tokenizer.decode(probe_ids) == probe_sample, "sample did not survive the round-trip"

# 3. the halves can be recovered, which is how a prompt is built at generation time
inner = probe_sample[len(START_OF_TEXT):-len(END_OF_TEXT)]
recovered = tuple(inner.split(SEPARATOR))
assert recovered == ("Riya", "hey, are you coming?"), recovered

print(f"{probe_sample!r}")
print(f"  -> {probe_ids}")
print(f"  {len(probe_ids)} tokens for {len(probe_sample)} characters, "
      f"{len(MARKERS)} of which are markers")

'<|startoftext|>Riya<|separator|>hey, are you coming?<|endoftext|>'
  -> [1024, 82, 105, 699, 1025, 104, 101, 121, 44, 473, 298, 764, 63, 1026]
  14 tokens for 65 characters, 3 of which are markers


## Running it on the real chats

`sorted()` again, for notebook 1's reason: glob order is filesystem-dependent, and an
unsorted read makes the saved dataset differ run to run for no reason.

**A sender name is not an identity.** Names repeat across exports — `Meera` appears in five
different chats here — so conditioning on `Meera` blends five people who share a first name.
That is fine for learning the *format*, which is what this dataset is for. If you want a
model that speaks as one specific person, point `CHAT_DIRECTORY` at a folder holding only
their chats.

In [9]:
chat_files = sorted(CHAT_DIRECTORY.glob("*.txt"))
if not chat_files:
    raise FileNotFoundError(f"no .txt exports found in {CHAT_DIRECTORY.resolve()}")

fine_tuning_data: list[str] = []
total_messages = 0

for file in chat_files:
    messages = read_chat(file)
    turns = group_by_sender(messages)          # per file, so a turn never spans two chats
    fine_tuning_data.extend(build_sample(sender, body) for sender, body in turns)
    total_messages += len(messages)

    print(f"{file.stem:<22} {len(messages):>5} messages -> {len(turns):>5} turns  "
          f"({len(messages) - len(turns):>4} merged)")

print(f"\n{total_messages:,} messages from {len(chat_files)} chats "
      f"-> {len(fine_tuning_data):,} samples")

book_club                386 messages ->   372 turns  (  14 merged)
bro_chat                 927 messages ->   906 turns  (  21 merged)
college_project          904 messages ->   861 turns  (  43 merged)
cousins_group            495 messages ->   482 turns  (  13 merged)
cricket_gang             471 messages ->   469 turns  (   2 merged)
dad_and_daughter         576 messages ->   471 turns  ( 105 merged)
exam_prep                498 messages ->   464 turns  (  34 merged)
family_group             917 messages ->   886 turns  (  31 merged)
foodie_chat              500 messages ->   407 turns  (  93 merged)
freelance_client         381 messages ->   304 turns  (  77 merged)
gym_buddies              367 messages ->   274 turns  (  93 merged)
highschool_couple        391 messages ->   319 turns  (  72 merged)
hostel_wing              471 messages ->   469 turns  (   2 merged)
landlord_tenant          496 messages ->   390 turns  ( 106 merged)
long_distance            502 messages ->   411 t

### Checking every sample, not a few

Two things can go wrong across the whole set, and both are quiet.

**A forged marker.** If a message body literally contained `<|endoftext|>`, its sample would
carry two of them and fine-tuning would read the tail as a second turn with no sender. This
is why minbpe defaults to `allowed_special="none_raise"` — encoding user text as control
tokens lets anyone plant a boundary. Counting markers per sample catches it. If it ever
fires, the fix is a decision (scrub the text? drop the message?) rather than something to
paper over here.

**A sample that does not survive encoding.** Checked on all of them rather than a spot
sample, because it costs a few seconds and the failure — one bad turn among twelve thousand
— is invisible any other way.

In [10]:
for sample in fine_tuning_data:
    counts = tuple(sample.count(marker) for marker in MARKERS)
    assert counts == (1, 1, 1), (
        f"expected one of each marker, got {dict(zip(MARKERS, counts))} "
        f"in {sample[:80]!r}"
    )

token_lengths = []
for sample in fine_tuning_data:
    ids = tokenizer.encode(sample, allowed_special="all")
    assert tokenizer.decode(ids) == sample, f"round-trip failed on {sample[:80]!r}"
    token_lengths.append(len(ids))

print(f"all {len(fine_tuning_data):,} samples carry exactly one of each marker "
      "and survive encode/decode")

all 11,719 samples carry exactly one of each marker and survive encode/decode


### How much fine-tuning signal is there?

Worth reading honestly, because the total is far bigger than the **195,469** tokens notebook
2 counted for the same messages — and none of the difference is new content. Every turn now
pays for three markers and for its sender's name, repeated on every turn that person takes.
The rest of the gap is re-tokenisation: each turn is encoded on its own, so merges no longer
run across message boundaries the way they did in one continuous stream.

The other number to watch is the longest sample against `block_size`. A sample longer than
the context window is cut off during fine-tuning, and what gets cut off is the end — the
`<|endoftext|>` this whole exercise exists to teach.

In [11]:
total_tokens = sum(token_lengths)
marker_tokens = len(MARKERS) * len(token_lengths)
sender_tokens = sum(
    len(tokenizer.encode(sample[len(START_OF_TEXT):].split(SEPARATOR, 1)[0]))
    for sample in fine_tuning_data
)

print(f"{len(fine_tuning_data):>9,} samples")
print(f"{total_tokens:>9,} tokens")
print(f"{total_tokens / len(token_lengths):>9.1f} tokens per sample on average "
      f"(shortest {min(token_lengths)}, longest {max(token_lengths)})\n")

for label, count in (
    ("markers", marker_tokens),
    ("sender names", sender_tokens),
    ("message text", total_tokens - marker_tokens - sender_tokens),
):
    print(f"  {label:<13}{count:>9,}  {count / total_tokens:>6.1%}")

too_long = sum(1 for n in token_lengths if n > BLOCK_SIZE)
print(f"\n{too_long} samples exceed block_size={BLOCK_SIZE} and would lose their ending")

   11,719 samples
  290,288 tokens
     24.8 tokens per sample on average (shortest 7, longest 179)

  markers         35,157   12.1%
  sender names    48,435   16.7%
  message text   206,696   71.2%

0 samples exceed block_size=256 and would lose their ending


## Saving the dataset

A JSON list of strings — the simplest thing that survives a round-trip without a schema.
`ensure_ascii=False` keeps emoji and non-Latin text readable in the file instead of turning
them into `\uXXXX` escapes; both forms load back to the same string, but only one of them
can be read in an editor.

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(
    json.dumps(fine_tuning_data, ensure_ascii=False, indent=4), encoding="utf-8"
)

print(f"wrote {OUTPUT_PATH.resolve()}")
print(f"  {len(fine_tuning_data):,} samples, {OUTPUT_PATH.stat().st_size:,} bytes")

wrote /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/TrainYourOwnLLM-Tutorial/output/fine_tuning/data/fine_tuning.json
  11,719 samples, 1,209,443 bytes


And read it back exactly as the fine-tuning notebook will. A dataset that is correct in
memory but not on disk breaks a notebook later, where the cause is invisible.

In [13]:
reloaded = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))

assert reloaded == fine_tuning_data, "the file does not match what was built in memory"
assert all(isinstance(sample, str) for sample in reloaded), "every sample must be a string"

print(f"reloaded {len(reloaded):,} samples, identical to the ones in memory\n")
print("first sample:")
print(f"  {reloaded[0]}")
print("\nlongest sample, truncated for display:")
print(f"  {reloaded[token_lengths.index(max(token_lengths))][:110]}...")

reloaded 11,719 samples, identical to the ones in memory

first sample:
  <|startoftext|>Meera<|separator|>happy new year everyone! shall we pick a book to start the year with something meaningful<|endoftext|>

longest sample, truncated for display:
  <|startoftext|>Naina<|separator|>sharing a recipe I want to try, sending the steps
paneer butter masala, step ...


### Where to go next

- **This dataset teaches format, not facts.** The model saw every one of these messages
  during pre-training. Fine-tuning on them installs the sender/turn structure — expect the
  loss to start below the pre-training loss and fall quickly, and do not read that as the
  model getting smarter.
- **Re-run this notebook if you change notebook 1's rules.** The two cleaning
  implementations have to stay identical, or fine-tuning drifts away from pre-training.
- **Expect it to overfit fast.** There is no held-out *content* here, only held-out samples,
  so watch validation loss the way notebook 4 does and stop early.
- **One voice beats twenty-three.** A dataset built from a single export is smaller but far
  more coherent, because the sender label then means one person rather than a name.